# exp02: LR × epochs grid, flat classifier, k = 0

Grid of 4 learning rates × 3 epoch counts, seed 42, everything else as exp01 (fp16 + fused AdamW, batch 32, k = 0).
Note: `tasks/C1-encoder-baseline/experiments/exp02_lr-epochs-grid/NOTE.md`.

The cell (2e-5, 5 epochs) is exp01's seed-42 run (`trial_flat_k0_seed42`, 0.4985), so it is not rerun: **11 runs**, ~5.2 GPU-hours, ~2.7 h with both T4s.

Temporary trial (`scripts/trial_run.py --mode flat`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU **T4 x2**: two independent runs at a time, one per GPU
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `smi-v4-meetings` dataset attached

The repo branch `C1/encoder-baseline` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C1/encoder-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [ ]:
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

## 4. Model loads in fp32
Also downloads the weights once (371 MB).

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
from smi.models.model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer("microsoft/deberta-v3-base")
print("dtype:", model.dtype)  # must be torch.float32
assert str(model.dtype) == "torch.float32"
del model

## 5. The grid
Change the values here if you want a different grid. Every run is one folder `exp02_lr<lr>_ep<epochs>_seed42`.

In [ ]:
SEED = 42
LRS = [1e-5, 2e-5, 3e-5, 5e-5]
EPOCHS = [3, 5, 8]
ALREADY_DONE = [(2e-5, 5)]  # exp01 seed 42 : same code , same seed , same settings

jobs = []
for lr in LRS:
    for epochs in EPOCHS:
        if (lr, epochs) in ALREADY_DONE:
            continue
        jobs.append((lr, epochs))

# longest runs first , so the two GPUs finish at about the same time
jobs = sorted(jobs, key=lambda job: job[1], reverse=True)
print(len(jobs), "runs:", jobs)

## 6. Run: a queue over both T4s
Each GPU takes the next run as soon as its current one ends. Each process sees only one GPU (`CUDA_VISIBLE_DEVICES`), so there is no communication between them. Progress: the newest line of each running log every 2 minutes.

**Safe to rerun after a restart:**
- a run whose folder has a `metrics.json` is finished and is skipped;
- a folder without one is an interrupted run: it is deleted and the run starts again.

In [ ]:
import shutil, subprocess, time

RUNS_DIR = "/kaggle/working/runs"
LOGS_DIR = "/kaggle/working/logs"
os.makedirs(LOGS_DIR, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_name(lr, epochs):
    return f"exp02_lr{lr:g}_ep{epochs}"


def start(gpu, lr, epochs):
    """one training process that sees only one GPU (CUDA_VISIBLE_DEVICES) ; its output goes to its own log file."""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    log_path = f"{LOGS_DIR}/{run_name(lr, epochs)}_gpu{gpu}.log"
    cmd = ["python", "scripts/trial_run.py", "--mode", "flat", "--seeds", str(SEED),
           "--lr", str(lr), "--epochs", str(epochs), "--name", run_name(lr, epochs),
           "--data-root", DATA_ROOT, "--model", "microsoft/deberta-v3-base", "--out-dir", RUNS_DIR]
    process = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env,
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    return process, log_path


# what is left to run
todo = []
for lr, epochs in jobs:
    run_dir = f"{RUNS_DIR}/{run_name(lr, epochs)}_seed{SEED}"
    if os.path.exists(f"{run_dir}/metrics.json"):
        print("finished already, skipped:", os.path.basename(run_dir))
        continue
    if os.path.exists(run_dir):
        print("interrupted run, deleted and started again:", os.path.basename(run_dir))
        shutil.rmtree(run_dir)
    todo.append((lr, epochs))

# running[gpu] = (process , log path) , or None when that GPU is free
running = {0: None, 1: None}
while len(todo) > 0 or any(job is not None for job in running.values()):
    for gpu in running:
        job = running[gpu]
        if job is not None and job[0].poll() is not None:
            process, log_path = job
            print(f"{os.path.basename(log_path)} finished with exit code {process.returncode}", flush=True)
            if process.returncode != 0:
                print(open(log_path).read()[-3000:], flush=True)  # the end of the log , to see the error
            running[gpu] = None
        if running[gpu] is None and len(todo) > 0:
            lr, epochs = todo.pop(0)
            running[gpu] = start(gpu, lr, epochs)
            print(f"GPU {gpu}: started {run_name(lr, epochs)}", flush=True)
    time.sleep(120)
    for gpu, job in running.items():
        if job is not None:
            lines = open(job[1]).read().strip().splitlines()
            print(f"{os.path.basename(job[1])}: {lines[-1] if lines else '(no output yet)'}", flush=True)
print("all runs finished")

## 7. Results
The grid of val macro-F1 (6), the selected epoch of every cell, and every run's per-epoch curve. A curve still rising at its last epoch is a hint that more epochs could help.

The (2e-5, 5) cell is exp01's seed 42: 0.4985, selected epoch 5.

In [ ]:
import json

results = {(2e-5, 5): (0.4985, 5)}  # exp01 seed 42
for lr in LRS:
    for epochs in EPOCHS:
        run_dir = f"{RUNS_DIR}/{run_name(lr, epochs)}_seed{SEED}"
        if not os.path.exists(f"{run_dir}/metrics.json"):
            continue
        m = json.load(open(f"{run_dir}/metrics.json"))
        results[(lr, epochs)] = (m["primary"]["macro_f1_6"], m["selected_epoch"])

# the grid : rows LR , columns epochs ; each cell "macro_f1_6 (selected epoch)"
print("val macro_f1_6 (selected epoch)")
print("lr      | " + " | ".join(f"{epochs} epochs".ljust(14) for epochs in EPOCHS))
for lr in LRS:
    cells = []
    for epochs in EPOCHS:
        if (lr, epochs) in results:
            score, selected = results[(lr, epochs)]
            cells.append(f"{score:.4f} ({selected})".ljust(14))
        else:
            cells.append("missing".ljust(14))
    print(f"{lr:<7g} | " + " | ".join(cells))

# every run : per-class F1 and the val curve per epoch
for lr in LRS:
    for epochs in EPOCHS:
        run_dir = f"{RUNS_DIR}/{run_name(lr, epochs)}_seed{SEED}"
        if not os.path.exists(f"{run_dir}/metrics.json"):
            continue
        m = json.load(open(f"{run_dir}/metrics.json"))
        print(f"\n=== {os.path.basename(run_dir)} | selected epoch {m['selected_epoch']} "
              f"| macro_f1_6 {m['primary']['macro_f1_6']:.4f} | macro_f1_7 {m['macro_f1_7']:.4f}")
        print("per-class F1:", {c: round(v["f1"], 3) for c, v in m["per_class"].items()})
        curve = []
        for line in open(f"{run_dir}/flat/history.jsonl"):
            h = json.loads(line)
            if h["type"] == "epoch":
                curve.append(f"{h['val_macro_f1']:.3f}")
                minutes = h["seconds"] / 60
        print("val macro F1 per epoch:", " , ".join(curve), f"| last epoch at {minutes:.1f} min")

## 8. Download the run folders
One zip with every exp02 run folder (settings, history, predictions, metrics) and the logs, in the notebook's output files.

In [ ]:
!cd /kaggle/working && zip -qr exp02_runs.zip runs/exp02_* logs && ls -lh exp02_runs.zip